# Qwen-Image-2.1 Colab Worker
Automated image generation and editing worker notebook executed via Colab CLI.

In [ ]:
import os, sys, subprocess

flag = "/content/.qwen_deps_installed"
if not os.path.exists(flag):
    print("Installing git transformers, diffusers, torchao, and accelerate...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "git+https://github.com/huggingface/transformers.git",
        "git+https://github.com/huggingface/diffusers.git",
        "torchao>=0.8.0",
        "huggingface_hub>=0.28.0",
        "accelerate",
        "qwen_vl_utils"
    ])
    with open(flag, "w") as f:
        f.write("ok")
    print("Environment initialized!")
else:
    print("Dependencies already initialized.")


In [ ]:
import torch

assert torch.cuda.is_available(), "GPU is required. Please allocate an L4, A100, or compatible GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

try:
    import PIL._typing
    if not hasattr(PIL._typing, "_Ink"):
        PIL._typing._Ink = getattr(PIL._typing, "Ink", tuple | list | str | int)
except Exception:
    pass

import diffusers
from diffusers import QwenImage21Pipeline
print(f"QwenImage21Pipeline ready! Diffusers version: {diffusers.__version__}")


In [ ]:
import os, time, torch
from diffusers import QwenImage21Pipeline

MODEL_ID = os.environ.get("QWEN_MODEL_ID", "Qwen/Qwen-Image-2.1")
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

if "QWEN_PIPE" not in globals():
    max_attempts = 5
    for attempt in range(1, max_attempts + 1):
        print(f"Loading {MODEL_ID} into GPU memory (attempt {attempt}/{max_attempts})...")
        try:
            QWEN_PIPE = QwenImage21Pipeline.from_pretrained(
                MODEL_ID,
                torch_dtype=DTYPE,
            ).to("cuda")
            print("Model loaded successfully into GPU memory!")
            break
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt == max_attempts:
                raise
            time.sleep(attempt * 4)
else:
    print("Reusing existing in-memory Qwen pipeline.")


In [ ]:
import json, random
from PIL import Image

mode = os.environ.get("QWEN_MODE", "t2i").lower()
prompt_file = os.environ.get("QWEN_PROMPT_FILE")
if prompt_file and os.path.exists(prompt_file):
    with open(prompt_file, "r", encoding="utf-8") as f:
        prompt = f.read().strip()
else:
    prompt = os.environ.get("QWEN_PROMPT", "A beautiful artwork")

output_path = os.environ.get("QWEN_OUTPUT_PATH", "/content/qwen_output.png")
width = int(os.environ.get("QWEN_WIDTH", "1024"))
height = int(os.environ.get("QWEN_HEIGHT", "1024"))
steps = int(os.environ.get("QWEN_STEPS", "40"))
seed_env = os.environ.get("QWEN_SEED")
seed = int(seed_env) if seed_env and seed_env.isdigit() else random.randint(1, 2**32 - 1)

print(f"Executing job: mode={mode}, steps={steps}, size={width}x{height}, seed={seed}")
print(f"Prompt: {prompt[:150]}...")

generator = torch.Generator(device="cuda").manual_seed(seed)

ref_images = []
refs_json = os.environ.get("QWEN_REFERENCE_IMAGES")
if refs_json:
    try:
        ref_paths = json.loads(refs_json)
        for p in ref_paths:
            if os.path.exists(p):
                ref_images.append(Image.open(p).convert("RGB"))
                print(f"Loaded reference image: {p}")
    except Exception as e:
        print(f"Warning: Failed to parse QWEN_REFERENCE_IMAGES: {e}")

pipe_kwargs = {
    "prompt": prompt,
    "num_inference_steps": steps,
    "generator": generator,
}

if width > 0 and height > 0:
    pipe_kwargs["width"] = width
    pipe_kwargs["height"] = height

if ref_images:
    pipe_kwargs["image"] = ref_images[0] if len(ref_images) == 1 else ref_images

print("Starting diffusion generation...")
result = QWEN_PIPE(**pipe_kwargs).images[0]

os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
result.save(output_path)
print(f"Generation completed successfully! Output saved to: {output_path}")
print(f"COLAB_CLI_OUTPUT_PATH: {output_path}")
